In [65]:
import pandas as pd
import numpy as np
import random
from collections import deque

In [ ]:
# Load the dataset
df = pd.read_csv('data/soc-sign-bitcoinotc.csv.gz', names=["SOURCE", "TARGET", "RATING", "TIME"])
df['TIME'] = pd.to_datetime(df['TIME'])

# Create neighbor mappings
outgoing_neighbors = df.groupby('SOURCE')['TARGET'].apply(set).to_dict()
incoming_neighbors = df.groupby('TARGET')['SOURCE'].apply(set).to_dict()
all_nodes = set(outgoing_neighbors.keys()).union(set(incoming_neighbors.keys()))

# Combine for undirected-like behavior
all_neighbors = {node: outgoing_neighbors.get(node, set()).union(incoming_neighbors.get(node, set()))
                 for node in all_nodes}

print(f"Graph contains {len(all_neighbors)} nodes.")

Graph contains 5881 nodes.


C:\Users\BartHolterman\AppData\Local\Temp\ipykernel_34392\2516860907.py:86: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  S = set(random.sample(all_nodes, num_initial_nodes))  # Initial sampled nodes


Algorithm converged after 1000 iterations.
Sampled subgraph contains 310 nodes.
Sampled subgraph contains 277 edges.


In [115]:
# Load the dataset
df = pd.read_csv('data/soc-sign-bitcoinotc.csv.gz', names=["SOURCE", "TARGET", "RATING", "TIME"])
df['TIME'] = pd.to_datetime(df['TIME'])

# Create directed neighbor mappings
outgoing_neighbors = df.groupby('SOURCE')['TARGET'].apply(set).to_dict()
incoming_neighbors = df.groupby('TARGET')['SOURCE'].apply(set).to_dict()
all_nodes = set(outgoing_neighbors.keys()).union(set(incoming_neighbors.keys()))

# Count directed edges
num_edges = len(df)

print(f"Directed graph contains {len(all_nodes)} nodes and {num_edges} directed edges.")

def icla_ns_directed(outgoing_neighbors, f, tau):
    """
    Implements the ICLA-NS algorithm for directed graphs.

    Parameters:
        outgoing_neighbors (dict): Node-to-outgoing-neighbors mapping.
        f (float): Fraction of nodes to sample.
        tau (float): Convergence threshold.

    Returns:
        sampled_nodes (set): The set of sampled nodes.
    """
    # Initialization
    num_nodes = len(outgoing_neighbors)
    num_sampled_nodes = int(f * num_nodes)
    sampled_nodes = set(random.sample(list(outgoing_neighbors.keys()), num_sampled_nodes))
    q = deque(sampled_nodes)  # Queue of sampled nodes

    # Automata mapping phase
    automata = {node: list(neighbors.union({node})) for node, neighbors in outgoing_neighbors.items()}
    state = {node: random.choice(automata[node]) for node in sampled_nodes}

    # Improvement phase
    while q:
        vi = q.popleft()
        automaton_vi = automata[vi]
        selected_action = state[vi]  # Use the current state as the action

        # Evaluate the selected action
        if selected_action in automaton_vi:  # Action must be valid (self or outgoing neighbor)
            if selected_action in sampled_nodes:
                # Reward: Keep the state
                pass
            else:
                # Penalize: Update state to another action
                state[vi] = random.choice(automaton_vi)
        else:
            # Penalize: Update state to another action
            state[vi] = random.choice(automaton_vi)

        # Add the selected action to the sampled set if valid
        if selected_action not in sampled_nodes:
            sampled_nodes.add(selected_action)
            q.append(selected_action)

        # Convergence check
        if len(sampled_nodes) / num_nodes >= tau:
            break

    return sampled_nodes

# Parameters
f = 0.5  # Fraction of nodes
tau = 0.05  # Convergence threshold

# Run the ICLA-NS algorithm for directed graphs
sampled_nodes = icla_ns_directed(outgoing_neighbors, f, tau)

# Count directed edges in the sampled subgraph
sampled_edges = sum(
    1 for u in sampled_nodes for v in outgoing_neighbors.get(u, set()) if v in sampled_nodes
)

print(f"Sampled {len(sampled_nodes)} nodes from the directed graph.")
print(f"Sampled subgraph contains {len(sampled_nodes)} nodes and {sampled_edges} directed edges.")


Directed graph contains 5881 nodes and 35592 directed edges.
Sampled 2407 nodes from the directed graph.
Sampled subgraph contains 2407 nodes and 8770 directed edges.
